# Hybrid GNN–LSTM Model: Portfolio PnL Simulation

**Subtitle:** End-to-end process from graph construction and temporal data to trained model and evaluation.

This tutorial showcases the **hybrid GNN–RNN** model in the QuantStrata library: a front-office–style pipeline with clear process descriptions, mathematical detail, and visualisations. We cover configuration, data engineering, model architecture, training, evaluation, deployment, and advanced topics.

In [ ]:
# Path setup: run from docs/tutorials/machine_learning/
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

# Output standardisation: single subfolder for this notebook
NOTEBOOK_SUBFOLDER = Path("../../..") / "hybrid_gnn_lstm"
CHECKPOINTS_DIR = NOTEBOOK_SUBFOLDER / "checkpoints"
LOGS_DIR = NOTEBOOK_SUBFOLDER / "logs"
MODEL_ARTIFACTS_DIR = NOTEBOOK_SUBFOLDER / "model_artifacts"
for d in (CHECKPOINTS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Imports
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

from src.machine_learning.models.gnn_rnn_hybrid import HybridGnnRnn, default_hybrid_model_config
from src.machine_learning.data.gnn_rnn_hybrid import build_gnn_data, GnnDataResult
from src.machine_learning.calibration.training_manager import TrainingManager, TrainingConfiguration

print("Key classes: HybridGnnRnn, build_gnn_data, TrainingManager, default_hybrid_model_config")

In [ ]:
# High-level flow diagram
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1, 1, figsize=(14, 3.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis("off")
boxes = [
    (1.2, 2, "Graph\n+ PnL history", "#e3f2fd"),
    (3, 2, "GNN\nblock", "#e8f5e9"),
    (4.5, 2, "RNN\nblock", "#e8f5e9"),
    (6, 2, "Fusion", "#fff3e0"),
    (7.5, 2, "Target\nattention", "#fce4ec"),
    (9, 2, "PnL\nprojection", "#f3e5f5"),
    (10.5, 2, "Predictions", "#e0f2f1"),
]
for x, y, label, color in boxes:
    rect = mpatches.FancyBboxPatch((x - 0.4, y - 0.3), 0.8, 0.6, boxstyle="round,pad=0.02",
                                    facecolor=color, edgecolor="gray", linewidth=1)
    ax.add_patch(rect)
    ax.text(x, y, label, ha="center", va="center", fontsize=9, fontweight="bold")
for i in range(len(boxes) - 1):
    ax.annotate("", xy=(boxes[i + 1][0] - 0.45, 2), xytext=(boxes[i][0] + 0.45, 2),
                arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
plt.title("Hybrid GNN–LSTM pipeline")
plt.tight_layout()
plt.show()

---
## 1. Configuration and Reproducibility

**Purpose:** Single place to define model and training config so runs are reproducible.

- **Model config:** `default_hybrid_model_config()` — GNN, RNN, fusion, attention, projection (units, layer types, dropout, etc.).
- **Training config:** `TrainingConfiguration` — epochs, batch_size, learning_rate, callbacks, paths (model_dir, early_stopping, reduce_lr_on_plateau).

**Key config keys:**
- `general.architecture`: `"default"` (full hybrid) vs `"rnn_only"` (baseline).
- GNN: `layer_type` — `"graph_sage"` or `"mixed_graph_sage"`.
- RNN: `layer_type` — `"lstm"`, `"bilstm"`, `"gru"`.
- Fusion: `fusion_mode` — `"gate"` or `"add"`.
- Projection: `baseline_new_mode`, `knn_mode`, `knn_k`, `knn_temperature` for new-trade generalisation.

In [ ]:
# Model and training config (used by TrainingManager and build_gnn_data)
n_targets = 10
model_config = default_hybrid_model_config(n_targets=n_targets)
training_config = TrainingConfiguration(
    name="hybrid_pnl_stage1",
    model="HybridGnnRnn",
    epochs=30,
    batch_size=32,
    learning_rate=1e-3,
    patience=15,
    model_dir=str(CHECKPOINTS_DIR),
    early_stopping=True,
    reduce_lr_on_plateau=True,
    reduce_lr_factor=0.5,
    reduce_lr_patience=5,
    loss="mse",
    metrics=["mae", "mse"],
)
print("Training: epochs=%s, batch_size=%s, model_dir=%s" % (
    training_config.epochs, training_config.batch_size, training_config.model_dir))

In [ ]:
# Build synthetic GNN data: train_ds, val_ds, proj_ds
data_result = build_gnn_data(
    use_synthetic=True,
    train_ratio=0.6,
    val_ratio=0.2,
    projection_ratio=0.2,
    batch_size=32,
    seed=42,
    n_trades=50,
    n_elementary=30,
    n_targets=n_targets,
    n_samples=500,
    n_timesteps=20,
    k_neighbours=5,
    noise_std=0.5,
)
train_ds, val_ds, proj_ds = data_result.train_ds, data_result.val_ds, data_result.proj_ds
metadata = data_result.metadata
# One batch for visualisation (cells below use `inputs`)
inputs, _ = next(iter(train_ds))
print("Metadata:", metadata)
print("train_ds element_spec (inputs):", train_ds.element_spec[0])

In [ ]:
# Visualisation: data shapes summary + sample PnL paths (2 samples, first 5 elementary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Shapes table (text)
ax = axes[0]
ax.axis("off")
shapes_text = (
    "n_trades: %s\nn_elementary: %s\nn_targets: %s\nn_samples: %s\nn_timesteps: %s\n"
    % (metadata.get("n_trades"), metadata.get("n_elementary"), metadata.get("n_targets"),
       metadata.get("n_samples"), metadata.get("n_timesteps"))
)
ax.text(0.1, 0.9, "Data shapes (metadata)\n" + shapes_text, transform=ax.transAxes,
        fontsize=11, verticalalignment="top", family="monospace")
# Sample PnL paths
ax = axes[1]
pnl = inputs["pnl_history"].numpy()
for s in range(min(2, pnl.shape[0])):
    for e in range(min(5, pnl.shape[2])):
        ax.plot(pnl[s, :, e], alpha=0.7, label=("S%d E%d" % (s, e) if s == 0 and e < 2 else None))
ax.set_xlabel("Timestep")
ax.set_ylabel("PnL")
ax.set_title("Sample PnL paths (batch, first 5 elementary)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Adjacency heatmap (first 20×20 trades)
adj = inputs["adjacency_matrix"].numpy()[0]
n_show = min(20, adj.shape[0])
plt.figure(figsize=(5, 4))
plt.imshow(adj[:n_show, :n_show], cmap="Blues", aspect="equal")
plt.colorbar(label="Adjacency (row-norm)")
plt.title("k-NN adjacency (first %d×%d trades)" % (n_show, n_show))
plt.xlabel("Trade j"); plt.ylabel("Trade i")
plt.tight_layout()
plt.show()

---
## 3. Model Architecture — Mathematical Description

**Purpose:** PhD-level formulation of each block and how they connect.

1. **GNN Block (GraphSAGE-style):** Aggregation $\mathbf{h}_{\mathcal{N}(v)} = \mathrm{Agg}(\{\mathbf{x}_u : u \in \mathcal{N}(v)\})$; update $\mathbf{h}_v^{(l+1)} = \sigma(\mathbf{W}_{\mathrm{self}}^{(l)} \mathbf{h}_v^{(l)} + \mathbf{W}_{\mathrm{neigh}}^{(l)} \mathbf{h}_{\mathcal{N}(v)}^{(l)})$; optional residual. Output: $\mathbf{H}_{\mathrm{gnn}} \in \mathbb{R}^{T \times d_g}$.

2. **RNN Block (LSTM):** Standard LSTM over PnL history; input $\mathbf{H}_{\mathrm{pnl}} \in \mathbb{R}^{B \times S \times N_e}$, output $\mathbf{h}_{\mathrm{rnn}} \in \mathbb{R}^{B \times d_r}$.

3. **Fusion (cross-attention + gating):** Query from RNN+GNN, Key/Value from GNN; scaled dot-product attention with **adjacency mask**; gate combines attention output with RNN. Output: $\mathbf{F}_{\mathrm{fused}} \in \mathbb{R}^{B \times T \times d_f}$.

4. **Target Attention:** Restrict to target indices; multi-head self-attention with adjacency masking; FFN + residual + LayerNorm. Output: $\mathbf{Z} \in \mathbb{R}^{B \times N_{\mathrm{tgt}} \times d_a}$.

5. **Projection (TargetPnlOutput):** Baseline (per-target kernel + bias) + residual MLP; for new targets, k-NN baseline in attribute space. Final: $\hat{y}_i = b_i + r_i$.

---
## 4. Training Pipeline

**Purpose:** Train with `TrainingManager`, including callbacks and logging.

- `TrainingManager(training_ds, model_config, validation_ds=val_ds, custom_callbacks=[...])`.
- `TrainingConfiguration`: epochs, batch_size, learning_rate, model_dir, early_stopping, reduce_lr_on_plateau.
- `manager.run(stages=[stage])` runs one or more stages; callbacks (ModelCheckpoint, EarlyStopping, ReduceLROnPlateau) are built inside `build_callbacks(stage)`.

**Loss:** MSE over $\hat{\mathbf{Y}}$ vs $\mathbf{Y}$: $\mathcal{L} = \frac{1}{BN_{\mathrm{tgt}}}\sum_{b,i}(y_{bi} - \hat{y}_{bi})^2$.

In [ ]:
# Create TrainingManager and run training (uses model_config and training_config from above)
manager = TrainingManager(train_ds, model_config, validation_ds=val_ds)
manager.run(stages=[training_config])

In [ ]:
# Training/validation loss curves
history = manager.training_history.get(training_config.name, {})
if history:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.get("loss", []), label="Train loss")
    ax.plot(history.get("val_loss", []), label="Val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Learning curves: Hybrid GNN–LSTM")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    plt.show()

---
## 5. Evaluation and Benchmarking

**Purpose:** Assess performance on train/val/projection splits; compare with RNN-only baseline if desired.

- Evaluate on `train_ds`, `val_ds`, `proj_ds`: aggregate MSE, MAE, RMSE, R², MAPE.
- **Predicted vs actual:** scatter with regression line; **residuals:** histogram, residual vs predicted, optional Q–Q.
- Compare **default** (full hybrid) vs **rnn_only** (same data) to show benefit of GNN + fusion.

In [ ]:
# Collect predictions and targets for each split
def collect_preds_and_targets(ds, model, max_batches=None):
    preds, tgts = [], []
    for i, (inp, t) in enumerate(ds):
        if max_batches is not None and i >= max_batches:
            break
        preds.append(model(inp, training=False))
        tgts.append(t)
    return tf.concat(preds, axis=0).numpy(), tf.concat(tgts, axis=0).numpy()

trained_model = manager.model
y_train_pred, y_train = collect_preds_and_targets(train_ds, trained_model)
y_val_pred, y_val = collect_preds_and_targets(val_ds, trained_model)
y_proj_pred, y_proj = collect_preds_and_targets(proj_ds, trained_model)

In [ ]:
# Regression metrics per split (run after "Collect predictions" cell below)
def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).flatten(), np.asarray(y_pred).flatten()
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mse)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - (ss_res / (ss_tot + 1e-12))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-12))) * 100
    return {"MSE": mse, "MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}

metrics_train = regression_metrics(y_train, y_train_pred)
metrics_val = regression_metrics(y_val, y_val_pred)
metrics_proj = regression_metrics(y_proj, y_proj_pred)
for name, m in [("Train", metrics_train), ("Val", metrics_val), ("Proj", metrics_proj)]:
    print("%s: MSE=%.4f MAE=%.4f RMSE=%.4f R²=%.4f MAPE=%.2f%%" % (name, m["MSE"], m["MAE"], m["RMSE"], m["R2"], m["MAPE"]))

fig, ax = plt.subplots(figsize=(8, 4))
splits = ["Train", "Val", "Proj"]
x = np.arange(len(splits))
width = 0.2
mets = [metrics_train, metrics_val, metrics_proj]
ax.bar(x - width, [m["MSE"] for m in mets], width, label="MSE")
ax.bar(x, [m["MAE"] for m in mets], width, label="MAE")
ax.bar(x + width, [m["R2"] for m in mets], width, label="R²")
ax.set_xticks(x); ax.set_xticklabels(splits); ax.legend(); ax.set_title("Metrics by split"); plt.tight_layout(); plt.show()

In [ ]:
# Collect predictions and targets for each split
def collect_preds_and_targets(ds, model, max_batches=None):
    preds, tgts = [], []
    for i, (inp, t) in enumerate(ds):
        if max_batches is not None and i >= max_batches:
            break
        preds.append(model(inp, training=False))
        tgts.append(t)
    return tf.concat(preds, axis=0).numpy(), tf.concat(tgts, axis=0).numpy()

trained_model = manager.model
y_train_pred, y_train = collect_preds_and_targets(train_ds, trained_model)
y_val_pred, y_val = collect_preds_and_targets(val_ds, trained_model)
y_proj_pred, y_proj = collect_preds_and_targets(proj_ds, trained_model)

In [ ]:
# Predicted vs actual (validation) + residuals
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Pred vs actual
ax = axes[0]
ax.scatter(y_val.flatten(), y_val_pred.flatten(), alpha=0.5, s=10)
lims = [min(y_val.min(), y_val_pred.min()), max(y_val.max(), y_val_pred.max())]
ax.plot(lims, lims, "k--", label="y = x")
ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
ax.set_title("Validation: Predicted vs actual")
ax.legend()
ax.grid(True)
# Residuals histogram
ax = axes[1]
resid = (y_val - y_val_pred).flatten()
ax.hist(resid, bins=30, edgecolor="black", alpha=0.7)
ax.axvline(0, color="red", linestyle="--")
ax.set_xlabel("Residual"); ax.set_ylabel("Count")
ax.set_title("Validation residuals")
plt.tight_layout()
plt.show()

---
## 6. Hyperparameter Sensitivity (Optional)

**Purpose:** Show how validation loss or R² changes with key knobs (e.g. gnn_units, rnn_units, dropout). Here we only illustrate the pattern with 2 short runs.

In [ ]:
# Optional: quick hyperparameter sensitivity (2 short runs; skip if short on time)
sensitivity_results = []
for gnn_units in [16, 32]:
    cfg = default_hybrid_model_config(n_targets=n_targets, gnn_units=gnn_units, rnn_units=32)
    mgr = TrainingManager(train_ds, cfg, validation_ds=val_ds)
    stage = TrainingConfiguration(name=f"gnn{gnn_units}", model="HybridGnnRnn", epochs=3, batch_size=32,
                                   learning_rate=1e-3, model_dir=str(CHECKPOINTS_DIR), early_stopping=False,
                                   reduce_lr_on_plateau=False)
    mgr.run(stages=[stage])
    hist = mgr.training_history.get(stage.name, {})
    sensitivity_results.append({"gnn_units": gnn_units, "val_loss": hist.get("val_loss", [None])[-1]})
print(sensitivity_results)
if sensitivity_results:
    plt.figure(figsize=(5, 3))
    plt.bar([r["gnn_units"] for r in sensitivity_results], [r["val_loss"] for r in sensitivity_results])
    plt.xlabel("gnn_units"); plt.ylabel("Val loss (last epoch)"); plt.title("Hyperparameter sensitivity (short run)"); plt.show()

---
## 7. Model Deployment and Inference

**Purpose:** Save/load model weights; run inference on new batches. Same input dict format; batched or single sample.

In [ ]:
# Save model weights to MODEL_ARTIFACTS_DIR
save_path = MODEL_ARTIFACTS_DIR / "hybrid_gnn_lstm.weights.h5"
manager.model.save_weights(str(save_path))
print("Saved weights to", save_path)

# Load into a new model (same config) and verify one batch matches
reloaded = HybridGnnRnn(model_config, name="hybrid_reloaded")
sample_inp, _ = next(iter(val_ds))
_ = reloaded(sample_inp, training=False)  # build
reloaded.load_weights(str(save_path))
pred_orig = manager.model(sample_inp, training=False)
pred_reload = reloaded(sample_inp, training=False)
print("Max diff after load:", np.abs(pred_orig.numpy() - pred_reload.numpy()).max())

---
## 8. Production Monitoring (Optional)

**Purpose:** Track latency and throughput for inference; optional drift over time.

- Time `model.predict()` for several batch sizes; report ms per batch and samples/sec.
- In production, monitor mean prediction or z-scores over time on a replay dataset to detect drift.

In [ ]:
# Inference latency and throughput (one batch from val_ds)
import time
inp, _ = next(iter(val_ds))
_ = manager.model(inp, training=False)  # warm
start = time.perf_counter()
for _ in range(50):
    _ = manager.model(inp, training=False)
elapsed = (time.perf_counter() - start) / 50
bs = inp["trade_features"].shape[0]
print("Batch size: %d | Latency: %.2f ms/batch | Throughput: %.0f samples/s" % (bs, elapsed * 1000, bs / elapsed))

In [ ]:
# Optional: compare with RNN-only baseline (same data)
config_rnn_only = {**model_config, "general": {**model_config["general"], "architecture": "rnn_only"}}
mgr_rnn = TrainingManager(train_ds, config_rnn_only, validation_ds=val_ds)
mgr_rnn.run(stages=[TrainingConfiguration(name="rnn_only", model="HybridGnnRnn", epochs=10, batch_size=32,
    learning_rate=1e-3, model_dir=str(CHECKPOINTS_DIR), early_stopping=True, patience=5)])
# Compare final val_loss: manager.training_history["hybrid_pnl_stage1"] vs mgr_rnn.training_history["rnn_only"]
print("Hybrid val_loss (from main run):", manager.training_history.get(training_config.name, {}).get("val_loss", [None])[-1])
print("RNN-only val_loss:", mgr_rnn.training_history.get("rnn_only", {}).get("val_loss", [None])[-1])

---
## Discussion: Why do the results look poor?

**Synthetic data:** The tutorial uses *synthetic* data. Targets are generated as a **linear combination of final timestep elementary PnL plus Gaussian noise** (`generate_targets` in `synthetic.py`). The weights are random and **do not depend on the graph or trade features**. So:

- The **graph (k-NN adjacency)** and **trade features** are not part of the data-generating process for targets.
- The model is asked to predict targets that are only loosely related to the PnL history; the GNN sees structure that is not used to create the labels.
- With high `noise_std`, the signal is weak; R² on val/proj can be low or even negative.

**Feature encoding and adjacency:** They are implemented correctly for the synthetic setup:

- **Trade features:** 7-d (moneyness, TTM, delta, vega, 3-d one-hot product). Used for k-NN and as model input.
- **Adjacency:** Row-normalised k-NN on trade features (Euclidean). Correct for "similar trades" structure.
- **Targets:** Linear in final elementary PnL + noise; elementary/target index split is random, so there is no built-in link between graph position and target.

**What would improve results:**

1. **Structured synthetic:** Make targets depend on graph (e.g. aggregate PnL over neighbours) or on trade features so the GNN has a learnable signal.
2. **Real FX data:** Use `build_fx_gnn_data(use_synthetic=False)` so targets come from real portfolio PnL; graph and features then reflect real relationships.
3. **More data / epochs:** Increase `n_samples`, `n_timesteps`, or train longer with careful regularisation.
4. **Lower noise:** Reduce `noise_std` in synthetic data to see better R² even with the current generative process.

The pipeline (config → data → model → training → evaluation → deployment) is correct; the apparent "rubbish" metrics are largely a consequence of the **synthetic data design**, not a bug in the model or encoding.